In [ ]:
# 필요한 패키지는 노트북의 현재 Python 커널에 설치합니다.
%pip install ultralytics opencv-python matplotlib roboflow

In [ ]:
from pathlib import Path
import os
import sys

# [수정 불필요] 현재 위치에서 프로젝트 폴더와 yolo_utiles.py를 자동으로 찾습니다.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "yolo_utiles.py").exists():
    PROJECT_DIR = PROJECT_DIR / "01.climbing_instanceseg"

if not (PROJECT_DIR / "yolo_utiles.py").exists():
    raise FileNotFoundError("yolo_utiles.py가 있는 프로젝트 폴더를 찾을 수 없습니다.")

# [수정 불필요] yolo_utiles.py를 import할 수 있도록 경로를 등록합니다.
sys.path.insert(0, str(PROJECT_DIR))

# [수정 불필요] 데이터셋과 실행 경로는 yolo_utiles.py에서 관리합니다.
from yolo_utiles import (
    DATA_YAML,
    DATASET_DIR,
    RUNS_DIR,
    YOLOSegmentPipeline,
)

print(f"Project: {PROJECT_DIR}")
print(f"Dataset: {DATASET_DIR}")

In [ ]:
# [필수 수정] 본인의 Roboflow API 키로 바꿉니다. 외부에 공개하지 마세요.
# [데이터셋 변경 시 수정] workspace, project, version을 변경합니다.

from roboflow import Roboflow

rf = Roboflow(api_key="여기에_본인의_Roboflow_API_키_입력")
project = rf.workspace("climbtag").project("climbing-hold-detection-7uehq")
version = project.version(29)

# [수정 불필요] yolo_utiles.py가 사용하는 dataset 폴더에 저장합니다.
dataset = version.download(
    "yolov11",
    location=str(DATASET_DIR),
)

print(f"Dataset downloaded to: {DATASET_DIR}")

In [ ]:
# [실험마다 수정] 사용할 모델 파일 경로를 입력합니다.
MODEL_PATH = input("모델 파일 경로를 입력하세요: ").strip()

# [실험마다 수정] runs에 저장할 실험 이름을 지정합니다.
EXPERIMENT_NAME = "yolo11_climbing_seg_vr1.1"

# [수정 불필요] 공통 경로와 파이프라인은 yolo_utiles.py에서 관리합니다.
pipeline = YOLOSegmentPipeline(
    model_path=MODEL_PATH,
    base_dir=RUNS_DIR,
    exp_name=EXPERIMENT_NAME,
)

# 실제 이미지와 라벨은 복사하지 않고, 실험에 사용한 데이터 설정만 보관합니다.
import shutil
import yaml

EXPERIMENT_DATASET_DIR = Path(pipeline.exp_dir) / "dataset"
EXPERIMENT_DATASET_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_DATA_YAML = EXPERIMENT_DATASET_DIR / "data.yaml"

with open(DATA_YAML, "r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

dataset_config["path"] = str(DATASET_DIR)
with open(EXPERIMENT_DATA_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dataset_config, file, allow_unicode=True, sort_keys=False)

for metadata_name in ("README.dataset.txt", "README.roboflow.txt"):
    metadata_path = DATASET_DIR / metadata_name
    if metadata_path.exists():
        shutil.copy2(metadata_path, EXPERIMENT_DATASET_DIR / metadata_name)

print(f"Experiment output: {pipeline.exp_dir}")
print(f"Experiment dataset config: {EXPERIMENT_DATA_YAML}")

In [ ]:
train_results = pipeline.train(
    data_yaml=str(EXPERIMENT_DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    resume=False,
)

print("Training complete")

In [ ]:
metrics = pipeline.validate(
    data_yaml=str(EXPERIMENT_DATA_YAML),
    split="val",
    device=0,
)

metrics

In [ ]:
# [실행할 때 수정] 추론할 이미지 또는 폴더 경로를 입력합니다.
PREDICT_SOURCE = input(
    "추론할 이미지 또는 폴더 경로를 입력하세요: "
).strip()

# [수정 불필요] 입력 경로가 실제로 존재하는지 확인합니다.
if not os.path.exists(PREDICT_SOURCE):
    raise FileNotFoundError(f"추론 대상 경로가 없습니다: {PREDICT_SOURCE}")

# [필요 시 수정] conf는 오탐지와 미탐지의 균형에 따라 조정할 수 있습니다.
prediction_results = pipeline.predict(
    source=PREDICT_SOURCE,
    conf=0.5,
    imgsz=640,
    device=0,
)

print(f"Predicted images: {len(prediction_results)}")
print(f"Saved results: {pipeline.predict_dir}")